# BirdCLEF 2026 — Perch v4 Track A Inference

Pipeline:
1. Load Perch v4 SavedModel (from dataset)
2. Run Perch on test soundscapes → embeddings + logits
3. Map logits to 234 competition species (direct Perch match + genus proxy for non-Aves)
4. Load pre-trained LogReg probes → blend 40% Perch + 60% probe
5. Class-type-aware temporal smoothing (texture avg-neighbour / event local-max)
6. File-max prior
7. Aggregate 5s windows → submission

In [ ]:
import glob
import os
import pickle
import tarfile

import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


# ── paths ────────────────────────────────────────────────────────────────────
def find_dir(patterns):
    for p in patterns:
        matches = glob.glob(p)
        if matches:
            return matches[0]
    raise FileNotFoundError(f"None of {patterns} found")


# Competition data
COMP_DIR = find_dir(
    [
        "/kaggle/input/birdclef-2026",
        "/kaggle/input/competitions/birdclef-2026",
    ]
)
print("Competition dir:", COMP_DIR)

# Perch v4 model tarball
PERCH_TARBALL = find_dir(
    [
        "/kaggle/input/perch-v4-model/perch_v4_model.tar.gz",
        "/kaggle/input/datasets/aldisued/perch-v4-model/*/perch_v4_model.tar.gz",
    ]
)
print("Perch tarball:", PERCH_TARBALL)

# Perch artifacts (probes + label list)
ARTIFACTS_DIR = find_dir(
    [
        "/kaggle/input/birdclef2026-perch-v4-artifacts",
        "/kaggle/input/datasets/aldisued/birdclef2026-perch-v4-artifacts/*",
    ]
)
print("Artifacts dir:", ARTIFACTS_DIR)

In [ ]:
# ── extract and load Perch v4 ────────────────────────────────────────────────
import time

PERCH_EXTRACT_DIR = "/tmp/perch_v4"
if not os.path.exists(PERCH_EXTRACT_DIR):
    print("Extracting Perch v4 model...")
    t0 = time.time()
    with tarfile.open(PERCH_TARBALL, "r:gz") as tar:
        tar.extractall("/tmp")
    # tarball extracts to /tmp/tfhub_modules/<hash>/
    # find the extracted dir
    extracted = glob.glob("/tmp/tfhub_modules/*/saved_model.pb")
    if extracted:
        PERCH_EXTRACT_DIR = os.path.dirname(extracted[0])
    print(f"Extracted in {time.time() - t0:.1f}s → {PERCH_EXTRACT_DIR}")
else:
    print(f"Already extracted: {PERCH_EXTRACT_DIR}")

print("Loading Perch v4 SavedModel...")
t0 = time.time()
perch_model = tf.saved_model.load(PERCH_EXTRACT_DIR)
print(f"Loaded in {time.time() - t0:.1f}s")

# Quick smoke test
test_audio = tf.zeros([1, 32000 * 5], dtype=tf.float32)
logits_test, emb_test = perch_model.infer_tf(test_audio)
print(f"Smoke test — logits: {logits_test.shape}, embeddings: {emb_test.shape}")

In [ ]:
# ── load species + label mappings ────────────────────────────────────────────
taxonomy = pd.read_csv(os.path.join(COMP_DIR, "taxonomy.csv"))
competition_species = taxonomy["primary_label"].astype(str).tolist()
n_species = len(competition_species)
sp_to_idx = {sp: i for i, sp in enumerate(competition_species)}
print(f"Competition species: {n_species}")

# Class type for smoothing
TEXTURE_CLASSES = {"Amphibia", "Insecta"}  # continuous callers
is_texture = np.array(
    [
        taxonomy.loc[taxonomy["primary_label"].astype(str) == sp, "class_name"].iloc[0]
        if sp in taxonomy["primary_label"].astype(str).values
        else "Aves" in TEXTURE_CLASSES
        for sp in competition_species
    ],
    dtype=bool,
)
# Simpler vectorized version:
label_to_class = dict(
    zip(taxonomy["primary_label"].astype(str), taxonomy["class_name"])
)
is_texture = np.array(
    [label_to_class.get(sp, "Aves") in TEXTURE_CLASSES for sp in competition_species]
)
is_event = ~is_texture
print(f"Texture species (Amphibia/Insecta): {is_texture.sum()}")
print(f"Event species (Aves + others): {is_event.sum()}")

# Load Perch label list
perch_labels = (
    open(os.path.join(ARTIFACTS_DIR, "perch_label_list.txt")).read().splitlines()
)
n_perch = len(perch_labels)
perch_to_idx = {lbl: i for i, lbl in enumerate(perch_labels)}
print(f"Perch v4 labels: {n_perch}")

# Direct mapping: competition species → Perch index
comp_to_perch = np.array(
    [perch_to_idx.get(sp, -1) for sp in competition_species], dtype=np.int32
)
perch_coverage = comp_to_perch >= 0
print(f"Direct Perch mapping: {perch_coverage.sum()} / {n_species}")

In [ ]:
# ── genus proxy for unmapped non-Aves species ────────────────────────────────
# For Amphibia/Insecta/Mammalia species not in Perch vocab:
# take max logit over all Perch labels sharing the same genus

# Build genus → list of Perch indices
genus_to_perch_indices = {}
for i, lbl in enumerate(perch_labels):
    # Perch labels are eBird codes (lowercase), not scientific names
    # Use first 6 chars as genus proxy for eBird codes is not reliable
    pass

# Better: use scientific name genus from taxonomy
# Perch labels are eBird codes; we need scientific names from a reference
# Since we don't have Perch's species metadata, fall back:
# unmapped species get 0.0 (Perch doesn't know them)

# Build genus mapping from taxonomy scientific names
genus_to_perch_indices = {}  # genus (lowercase) → list of Perch indices
for i, lbl in enumerate(perch_labels):
    # eBird codes: 6-char codes don't map to genus
    # Use first 3 chars as a rough genus proxy
    prefix = lbl[:3].lower()
    genus_to_perch_indices.setdefault(prefix, []).append(i)

# For competition species: extract genus from scientific_name
taxonomy["genus"] = taxonomy["scientific_name"].str.split().str[0].str.lower()

# Build per-species genus proxy index list
sp_genus_perch = {}  # sp → list of Perch indices for genus
for _, row in taxonomy.iterrows():
    sp = str(row["primary_label"])
    if comp_to_perch[sp_to_idx[sp]] >= 0:
        continue  # has direct mapping
    genus = row["genus"]
    # Try first 3 chars of genus
    prefix = genus[:3].lower() if len(genus) >= 3 else genus.lower()
    indices = genus_to_perch_indices.get(prefix, [])
    if indices:
        sp_genus_perch[sp] = indices

print(
    f"Unmapped species with genus proxy: {len(sp_genus_perch)} / {(~perch_coverage).sum()}"
)
print(
    f"Unmapped species without proxy: {(~perch_coverage).sum() - len(sp_genus_perch)}"
)

In [ ]:
# ── load LogReg probes ───────────────────────────────────────────────────────
with open(os.path.join(ARTIFACTS_DIR, "perch_probes.pkl"), "rb") as f:
    probe_data = pickle.load(f)

pca = probe_data["pca"]
scaler = probe_data["scaler"]
probes = probe_data["probes"]  # dict: sp_label → LogisticRegression or None
probe_species = set(probe_data["probe_species"])
probe_species_arr = probe_data["species"]  # (234,) in same order as competition_species

print(
    f"Probes loaded: {len([v for v in probes.values() if v is not None])} trained probes"
)
print(f"Species order matches: {list(probe_species_arr) == competition_species}")

In [ ]:
# ── inference config ─────────────────────────────────────────────────────────
SR = 32_000
CLIP_SAMPLES = SR * 5
PROBE_WEIGHT = 0.6  # weight for LogReg probe predictions
PERCH_WEIGHT = 0.4  # weight for raw Perch predictions
ALPHA_TEXTURE = 0.35  # average-neighbour smoothing for texture classes
ALPHA_EVENT = 0.15  # local-max propagation for event classes
FILE_MAX_WEIGHT = 0.05


def predict_window(chunk: np.ndarray):
    """Run Perch on a single 5s chunk. Returns (perch_probs (234,), probe_probs (234,))."""
    x = tf.constant(chunk[np.newaxis], dtype=tf.float32)
    raw_logits, emb = perch_model.infer_tf(x)
    raw_logits = raw_logits[0].numpy()  # (n_perch,)
    emb = emb[0].numpy()  # (1280,)

    # --- Perch probs in competition space ---
    comp_probs = np.zeros(n_species, dtype=np.float32)
    # Direct mapped species
    comp_probs[perch_coverage] = 1.0 / (
        1.0 + np.exp(-raw_logits[comp_to_perch[perch_coverage]])
    )
    # Genus proxy for unmapped species
    for sp, indices in sp_genus_perch.items():
        sp_idx = sp_to_idx[sp]
        max_logit = raw_logits[indices].max()
        comp_probs[sp_idx] = 1.0 / (1.0 + np.exp(-max_logit))

    # --- Probe probs ---
    emb_scaled = scaler.transform(emb[np.newaxis])  # (1, 1280)
    emb_pca = pca.transform(emb_scaled)  # (1, 64)
    probe_probs = np.zeros(n_species, dtype=np.float32)
    for sp_idx, sp in enumerate(competition_species):
        clf = probes.get(sp)
        if clf is not None:
            probe_probs[sp_idx] = clf.predict_proba(emb_pca)[0, 1]
        else:
            probe_probs[sp_idx] = comp_probs[sp_idx]  # fall back to Perch

    # Blend
    blended = PERCH_WEIGHT * comp_probs + PROBE_WEIGHT * probe_probs
    return blended


def smooth_preds(preds: np.ndarray) -> np.ndarray:
    """Class-type-aware temporal smoothing. preds: (T, n_species)"""
    T = preds.shape[0]
    smoothed = preds.copy()
    for t in range(T):
        prev = preds[t - 1] if t > 0 else preds[t]
        nxt = preds[t + 1] if t < T - 1 else preds[t]
        # Texture: average-neighbour
        smoothed[t, is_texture] = (
            (1 - ALPHA_TEXTURE) * preds[t, is_texture]
            + (ALPHA_TEXTURE / 2) * prev[is_texture]
            + (ALPHA_TEXTURE / 2) * nxt[is_texture]
        )
        # Event: local-max propagation
        local_max = np.maximum(preds[t], np.maximum(prev, nxt))
        smoothed[t, is_event] = (1 - ALPHA_EVENT) * preds[
            t, is_event
        ] + ALPHA_EVENT * local_max[is_event]
    return smoothed


print("Inference functions defined.")

In [ ]:
# ── find test soundscapes ────────────────────────────────────────────────────
test_soundscape_dir = os.path.join(COMP_DIR, "test_soundscapes")
test_files = sorted(glob.glob(os.path.join(test_soundscape_dir, "*.ogg")))
print(f"Test soundscapes: {len(test_files)}")

# Load sample submission for row ordering
sample_sub = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))
print(f"Sample submission rows: {len(sample_sub)}")
print(sample_sub.head(3))

In [ ]:
# ── main inference loop ──────────────────────────────────────────────────────
all_rows = []
t_start = time.time()

for i, sc_path in enumerate(test_files):
    stem = os.path.splitext(os.path.basename(sc_path))[0]
    try:
        y, _ = librosa.load(sc_path, sr=SR, mono=True)
    except Exception as e:
        print(f"WARN: {sc_path}: {e}")
        continue

    n_slots = len(y) // CLIP_SAMPLES
    if n_slots == 0:
        continue

    # Run Perch on each 5s window (non-overlapping for speed)
    slot_preds = np.zeros((n_slots, n_species), dtype=np.float32)
    for slot in range(n_slots):
        chunk = y[slot * CLIP_SAMPLES : (slot + 1) * CLIP_SAMPLES]
        slot_preds[slot] = predict_window(chunk)

    # Temporal smoothing
    slot_preds = smooth_preds(slot_preds)

    # File-max prior
    file_max = slot_preds.max(axis=0, keepdims=True)
    slot_preds = slot_preds + FILE_MAX_WEIGHT * file_max
    slot_preds = np.clip(slot_preds, 0.0, 1.0)

    # Build rows
    for slot in range(n_slots):
        end_sec = (slot + 1) * 5
        row_id = f"{stem}_{end_sec}"
        row = {"row_id": row_id}
        for sp_idx, sp in enumerate(competition_species):
            row[sp] = float(slot_preds[slot, sp_idx])
        all_rows.append(row)

    if (i + 1) % 20 == 0 or i == 0:
        elapsed = time.time() - t_start
        rate = (i + 1) / elapsed
        eta = (len(test_files) - i - 1) / max(rate, 1e-9)
        print(
            f"[{i + 1}/{len(test_files)}] {rate:.2f} files/s | ETA {eta / 60:.1f} min",
            flush=True,
        )

elapsed_total = time.time() - t_start
print(f"\nDone in {elapsed_total / 60:.1f} min. Rows generated: {len(all_rows)}")

In [ ]:
# ── build submission ─────────────────────────────────────────────────────────
sub_df = pd.DataFrame(all_rows)

# Reindex to match sample submission row order
sub_df = sub_df.set_index("row_id")
sub_df = sub_df.reindex(sample_sub["row_id"])

# Fill any missing rows with 0
sub_df = sub_df.fillna(0.0)
sub_df = sub_df.reset_index()

# Ensure column order matches sample submission
sub_df = sub_df[["row_id"] + [c for c in sample_sub.columns if c != "row_id"]]

print(f"Submission shape: {sub_df.shape}")
print(sub_df.head(3))
sub_df.to_csv("submission.csv", index=False)
print("Saved submission.csv")